### Simple Chain
---
A 3 steps chain:
1. Take input from user.
2. Pass the input to a LLM.
3. Show the LLM's output to the user.

In [33]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import PromptTemplate

In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()

HUGGINGFACE_API_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HUGGINGFACE_API_TOKEN

In [37]:
# make the prompt
prompt = PromptTemplate(
    template = "Generate 5 interesting fact about {topic}",
    input_variables = ['topic']
)

In [38]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.2-1B-Instruct",
    task="text-generation",
    # huggingfacehub_api_token = HUGGINGFACE_API_TOKEN
)
model = ChatHuggingFace(llm = llm)

In [39]:
# make a string parser
from langchain_core.output_parsers import StrOutputParser
sting_parser = StrOutputParser()

In [40]:
# form the chain
simple_chain = prompt | model | sting_parser

In [41]:
user_topic = input("Enter a topic name: ")
output = simple_chain.invoke({
    "topic": user_topic
})
print(output)

Here are 5 interesting facts about football:

1. **The first-ever golf ball was made from a mixture of clay, sawdust, and feathers**: Before the advent of modern footballs, golf balls were made by blowing air through a mixture of clay, sawdust, and feathers. This mixture would be shaped and molded into a ball, and then air would be blown through it to give it weight and air pressure.

2. **The shortest player in NFL history was 5'1" (1.55m)**: In 2014, a player named Chris Rogers played for the Tampa Bay Buccaneers. Despite his small stature, Rogers was 5'1" (1.55m) and weighed 135 pounds (61.2kg). He played as a wide receiver.

3. **The longest recorded football game lasted 4 hours and 48 minutes**: In 1966, a game between the Dallas Cowboys and the Minnesota Vikings ended in a tie after 4 hours and 48 minutes. The game was played in the snow, and the teams were given a 30-minute halftime break.

4. **The first-ever football trick play was the "Tuck Rule"**: In 2002, the New England P

In [42]:
simple_chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
   +-----------------+     
   | ChatHuggingFace |     
   +-----------------+     
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


**Building a 2 step sequential Chain**

- Take a topic from user.
- Make a prompt to generate a detailed report on it.
- Pass the topic to a LLM and generate a detailed report on it.
- Make a 2nd prompt to get 5 most import point from the detailed report.
- Send it to the same LLM and get the key 5 points.

In [43]:
# define the prompt
prompt1 = PromptTemplate(
    template = "Generate a detailed report on this {topic}.",
    input_variables = ["topic"]
)
prompt2 = PromptTemplate(
    template = "Give me the keywords from the given report here.\n {report}",
    input_variables = ['report']
)

In [45]:
# define 
from langchain_core.output_parsers import StrOutputParser
str_parser = StrOutputParser()

In [44]:
model

ChatHuggingFace(llm=HuggingFaceEndpoint(repo_id='meta-llama/Llama-3.2-1B-Instruct', huggingfacehub_api_token='REMOVED_SECRET', stop_sequences=[], server_kwargs={}, model_kwargs={}, model='meta-llama/Llama-3.2-1B-Instruct', client=<InferenceClient(model='meta-llama/Llama-3.2-1B-Instruct', timeout=120)>, async_client=<InferenceClient(model='meta-llama/Llama-3.2-1B-Instruct', timeout=120)>, task='text-generation'), model_id='meta-llama/Llama-3.2-1B-Instruct', temperature=0.8, top_p=0.95, max_tokens=512, model_kwargs={})

In [47]:
from langchain_core.runnables import RunnableLambda

In [48]:
# define the chain
chain1 = (
    prompt1| 
    model| 
    str_parser| 
    RunnableLambda(lambda x: {"report" : x})|
    prompt2| 
    model|
    str_parser
)

In [49]:
user_topic = "Unemployment issue in Bangladesh."
result = chain1.invoke({
    "topic": user_topic
})

In [50]:
type(result)

langchain_core.messages.base.TextAccessor

In [52]:
print(result)

The keywords from the given report are:

1. Unemployment
2. Bangladesh
3. Report
4. Population
5. Causes
6. Effects
7. Solutions
8. Informal Economy
9. Infrastructure
10. Aging Population
11. Male-Male Fertility Rate
12. Urbanization
13. Productivity
14. Poverty
15. Social Issues
16. Cultural Impact
17. Economic Growth
18. National Income
19. Labor Market
20. Economic Development

These keywords can be used for information retrieval, data analysis, or further research on the topic of unemployment in Bangladesh.


In [53]:
# visualize the chain1
chain1.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
   +-----------------+     
   | ChatHuggingFace |     
   +-----------------+     
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
        +--------+         
        | Lambda |         
        +--------+         
            *              
            *              
            *       

### Building a Parallel Chain.
* Take a long detailed text(explaination of something).
* Parallel Part:
   - Send to model-1 to generate notes from the doc.
   - Send to model-2 to generate quiz(mcq) from the doc.
* Take the note and quiz then send these to a 3rd model to combine these.

In [109]:
model_for_note = ChatHuggingFace(
    llm = HuggingFaceEndpoint(
        repo_id = "meta-llama/Llama-3.2-1B-Instruct",
        task = "text-generation",
        max_new_tokens=1024
    )
)
model_for_quiz = ChatHuggingFace(
    llm = HuggingFaceEndpoint(
        repo_id="microsoft/Phi-3-mini-4k-instruct",
        task = "text-generation",
        max_new_tokens=1024
    )
)

In [110]:
prompt_for_note = PromptTemplate(
    template = "Generate short and simple note from the following text: \n {text}",
    input_variables = ['text']
)
prompt_for_quiz = PromptTemplate(
    template = "Generate few multiple choice questions from the following text: \n{text}",
    input_variables = ['text']
)
prompt_for_merge_note_quiz = PromptTemplate(
    template = """Merge the note and multiple choice question into a single document.\n
    Note: {note} \n Multiple-Choice Questions: {questions}
    """,
    input_variables = ["note" , "questions"]
)

In [111]:
# define the parser
str_parser

StrOutputParser()

In [112]:
from langchain_core.runnables import RunnableParallel

In [118]:
# Define 2 sequential chain
note_chain = prompt_for_note | model_for_note | str_parser
quiz_chain = prompt_for_quiz | model | str_parser

In [119]:
# now parallel these 2 sequential chain  in parallel
parallel_chain = RunnableParallel(
    note = note_chain,
    questions = quiz_chain
)

In [120]:
# marge with final model
final_chain = parallel_chain | prompt_for_merge_note_quiz | model | str_parser

In [121]:
with open("doc.txt" , "r") as f:
    doc = f.read()

doc

'RandomForestClassifier\nclass sklearn.ensemble.RandomForestClassifier(n_estimators=100, *, criterion=\'gini\', max_depth=None, min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_features=\'sqrt\', max_leaf_nodes=None, min_impurity_decrease=0.0, bootstrap=True, oob_score=False, n_jobs=None, random_state=None, verbose=0, warm_start=False, class_weight=None, ccp_alpha=0.0, max_samples=None, monotonic_cst=None)[source]\nA random forest classifier.\n\nA random forest is a meta estimator that fits a number of decision tree classifiers on various sub-samples of the dataset and uses averaging to improve the predictive accuracy and control over-fitting. Trees in the forest use the best split strategy, i.e. equivalent to passing splitter="best" to the underlying DecisionTreeClassifier. The sub-sample size is controlled with the max_samples parameter if bootstrap=True (default), otherwise the whole dataset is used to build each tree.\n\nFor a comparison between tree-based

In [122]:
result = final_chain.invoke({
    "text": doc
})

In [123]:
print(result)

Here's the merged document:

Random Forest Classifier

**Note**

Random Forest Classifier is a meta estimator used for predicting class labels. It fits a number of decision tree classifiers on various sub-samples of the dataset and uses averaging to improve the predictive accuracy and control over-fitting. The estimator has native support for missing values (NaNs) and uses the best split strategy to reduce overfitting.

**Multiple-Choice Questions**

1. What is the native support for missing values in Random Forest Classifier?
a) NaNs
b) None
c) All missing values are mapped to the most abundant feature.
d) Samples with multiple missing values are mapped to the most frequent feature.

Answer: d) Samples with multiple missing values are mapped to the most frequent feature.

2. Which parameter controls the behavior of trees in the forest when it comes to splitting nodes?
a) max_depth
b) min_samples_split
c) min_samples_leaf
d) min_weight_fraction_leaf

Answer: b) min_samples_split

3. Ho

In [124]:
final_chain.get_graph().print_ascii()

            +-------------------------------+            
            | Parallel<note,questions>Input |            
            +-------------------------------+            
                  ***               ***                  
               ***                     ***               
             **                           **             
+----------------+                    +----------------+ 
| PromptTemplate |                    | PromptTemplate | 
+----------------+                    +----------------+ 
          *                                   *          
          *                                   *          
          *                                   *          
+-----------------+                  +-----------------+ 
| ChatHuggingFace |                  | ChatHuggingFace | 
+-----------------+                  +-----------------+ 
          *                                   *          
          *                                   *          
          *   

### Conditional Chain

In [135]:
llm = HuggingFaceEndpoint(
    repo_id = "Qwen/Qwen2.5-7B-Instruct",
    task = "text-generation",
    temperature = 0.1,
    max_new_tokens = 512
)
model = ChatHuggingFace(llm = llm)

In [136]:
from pydantic import BaseModel, Field
from typing import Literal

class Sentiment(BaseModel):
    sentiment_type: Literal['positive' , 'negative'] = Field(
        description = "Give the sentiment of the feedback"
    )

In [137]:
from langchain_core.output_parsers import PydanticOutputParser
pydantic_parser = PydanticOutputParser(
    pydantic_object = Sentiment
)

In [139]:
prompt1 = PromptTemplate(
    template = """Classify the sentiment of the following feedback text
    into only either positive or negative in the following format.\n {feedback}\n
    Format: {format_instruction}""",
    input_variables = ['feedback'],
    partial_variables = {
        "format_instruction" : pydantic_parser.get_format_instructions()
    }
)

classifier = prompt1 | model | pydantic_parser

In [140]:
review_text = """
Sarah here. Absolutely love this coffee maker! Brews perfectly every time.
Super easy to clean and looks great on the counter. 
The only downside is it's quite loud and the carafe lid is flimsy.
"""
print(
    classifier.invoke({
        "feedback": review_text
    })
)

sentiment_type='positive'


In [141]:
sentiment_result = classifier.invoke({
    "feedback": review_text
})

In [142]:
type(sentiment_result)

__main__.Sentiment

In [143]:
sentiment_result.sentiment_type

'positive'

In [145]:
# Step 1: carry feedback alongside sentiment result
from langchain_core.runnables import RunnablePassthrough
classifier_chain = RunnablePassthrough.assign(
    sentiment = lambda x: classifier.invoke({"feedback": x["feedback"]})
)

In [144]:
pos_prompt = PromptTemplate(
    template = "Write an appropriate email for this positive feedback:\n{feedback}",
    input_variables = ["feedback"]
)
neg_prompt = PromptTemplate(
    template = "Write an appropriate email for this negative feedback:\n{feedback}",
    input_variables = ["feedback"]
)

In [146]:
# now based on the sentiment make 2 branch: +ve and -ve branch
from langchain_core.runnables import RunnableBranch

# Step 2: branch based on sentiment, pass feedback to prompts
branch_chain = RunnableBranch(
    (
        lambda x: x["sentiment"].sentiment_type == "positive",
        RunnableLambda(lambda x: {"feedback": x["feedback"]}) | pos_prompt | model | str_parser
    ),
    (
        lambda x: x["sentiment"].sentiment_type == "negative",
        RunnableLambda(lambda x: {"feedback": x["feedback"]}) | neg_prompt | model | str_parser
    ),
    RunnableLambda(lambda x: "Could not find any sentiment.")
)

In [147]:
# Step 3: full chain
full_chain = classifier_chain | branch_chain

In [148]:
result = full_chain.invoke({"feedback": review_text})
print(result)

Subject: Positive Feedback on Your Coffee Maker

Dear [Company Name] Team,

I hope this email finds you well.

I wanted to take a moment to express my satisfaction with the coffee maker I recently purchased from your store. I am absolutely loving it! The coffee it brews is always perfect, and I appreciate how easy it is to clean. Additionally, the sleek design makes it a great addition to my kitchen counter.

However, I do have a couple of minor concerns. The coffee maker is quite loud when in use, which can be a bit disruptive in the morning. Also, the carafe lid feels a bit flimsy and I worry about it potentially breaking.

Thank you for your attention to this feedback. I hope it helps improve the product for future customers.

Best regards,

Sarah
